In [1]:
from google.colab import drive
drive.mount('/content/drive')
%pip install lime -q

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer
import os

MODEL_PATH = '/content/drive/MyDrive/DrugRadar/models/checkpoints/best_model'
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

def predict_proba(texts):
    results = []
    for text in texts:
        enc = tokenizer(text, return_tensors='pt', truncation=True,
                        max_length=128, padding=True).to(device)
        with torch.no_grad():
            logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
        results.append(probs)
    return np.array(results)

Loading weights:   0%|          | 0/201 [00:02<?, ?it/s]

In [3]:
os.makedirs('/content/drive/MyDrive/DrugRadar/docs', exist_ok=True)
explainer = LimeTextExplainer(class_names=['No ADE', 'ADE'])

test_texts = [
    "I developed severe stomach bleeding after taking ibuprofen daily for a month.",
    "The medication worked well with no side effects at all.",
    "My ears have been ringing nonstop since I started this drug lol",
    "Doctor prescribed this and it has been helping my condition.",
    "Gained 20 pounds in 2 months on this medication, constant fatigue and nausea.",
    "This drug completely resolved my symptoms with zero problems.",
]

for i, text in enumerate(test_texts):
    exp = explainer.explain_instance(text, predict_proba, num_features=8, num_samples=150)
    exp.save_to_file(f'/content/drive/MyDrive/DrugRadar/docs/lime_{i+1}.html')
    print(f'Done {i+1}: {exp.as_list()[:3]}')

print('All LIME explanations saved!')

Done 1: [(np.str_('ibuprofen'), 0.8371343903775592), (np.str_('stomach'), 0.17094427379532184), (np.str_('bleeding'), 0.1683480165556789)]
Done 2: [(np.str_('The'), -2.9165585347722312e-05), (np.str_('all'), -2.473250249227244e-05), (np.str_('no'), -2.0841986082456773e-05)]
Done 3: [(np.str_('started'), -0.0009896423767800193), (np.str_('lol'), 0.000876230285502933), (np.str_('drug'), -0.0008029922665318974)]
Done 4: [(np.str_('condition'), -3.099953485755773e-05), (np.str_('Doctor'), -1.459334876399183e-05), (np.str_('prescribed'), -1.2527004869362882e-05)]
Done 5: [(np.str_('constant'), -3.736146241954794e-05), (np.str_('medication'), -3.314497693338379e-05), (np.str_('and'), -1.8230543742883215e-05)]
Done 6: [(np.str_('problems'), -4.19357166880943e-05), (np.str_('symptoms'), -3.644067626698175e-05), (np.str_('resolved'), -3.1838301180895486e-05)]
All LIME explanations saved!
